# Fine-tuning the curator — QLoRA on Colab

Trains a LoRA adapter for **one job**: given a task and a list of candidate memories, return the ids of the relevant ones. Nothing else in `one-for-all` is fine-tuned, and that is deliberate — see [MODEL.md](../MODEL.md) §1.2 for why the critic is bought rather than trained.

## Read this before running anything

Fine-tuning is **step 4 of 4** in MODEL.md §7's ladder. The three cheaper rungs come first, and they usually win:

| Step | Cost | Typical gain |
|---|---|---|
| 1. Better prompt | minutes | most of it |
| 2. Few-shot examples from the decision log | free, self-improving | often closes the gap entirely |
| 3. Better base model | a config change | sometimes large |
| 4. QLoRA (this notebook) | an afternoon | the remainder |

And the hard gate:

> Reaching step 4 without a scoreboard means you cannot tell whether it worked. That is not a fine-tune, it is a ritual.

So this notebook **evaluates before it trains**, and refuses to declare success on anything but a measured improvement. If the before/after numbers are a wash, the correct outcome is to throw the adapter away. That is a successful run of this notebook.

## What you need

- **Runtime → Change runtime type → T4 GPU** (free tier is enough; a 4B QLoRA fits in ~6 GB)
- `train.jsonl` — from `one-for-all-cli export train.jsonl`
- `eval.jsonl` — from `one-for-all-cli seed-eval eval.jsonl`, **hand-corrected**

The eval file is the one that takes real work. The seeded version prefills `expected` with what the curator already chose, so an uncorrected file agrees with the model by construction and measures nothing.

---
## 1. Environment

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU."
)
props = torch.cuda.get_device_properties(0)
print(f"{props.name}, {props.total_memory / 1e9:.1f} GB VRAM")

# MODEL.md §7: 8-12 GB suffices for a 7-8B QLoRA; a 4B curator needs far less.
if props.total_memory < 8e9:
    print("WARNING: under 8 GB. Drop MAX_SEQ_LENGTH and BATCH_SIZE below.")

In [ ]:
%%capture
# Unsloth is the fast path on consumer GPUs (MODEL.md §7 toolchain ladder:
# Unsloth -> Axolotl -> TRL). It pins compatible trl/peft/bitsandbytes.
!pip install -q unsloth
!pip install -q --no-deps --upgrade unsloth_zoo

In [ ]:
# The eval must score the PRODUCTION parser, not a copy of it. A notebook-local
# reimplementation would report a format-compliance number the running daemon
# never achieves — so we install the package itself.
#
# Set this to your repo. If it is private, instead build a wheel locally with
#   python -m build
# and upload the .whl in the next cell.
REPO = "https://github.com/supermeet/one-for-all"

try:
    !pip install -q "git+{REPO}"
    from one_for_all.curator import SYSTEM_PROMPT, parse_selection
    from one_for_all import evals
    print("installed from git")
except Exception as exc:
    print(f"git install failed ({exc}).")
    print("Upload a wheel built with `python -m build`, then:")
    print("  !pip install -q /content/one_for_all-*.whl")
    raise

---
## 2. Data

Upload `train.jsonl` and `eval.jsonl`.

MODEL.md §7 on volume: **500–2,000 examples, and quality dominates.** 500 clean beat 5,000 noisy; 1,000 hand-curated beat 100,000 noisy. `one-for-all-cli export` already applied the §7 filters — dropped ambiguous tasks, dropped duplicates, kept every recorded miss, balanced the selection sizes.

In [ ]:
from google.colab import files

print("Upload train.jsonl and eval.jsonl")
files.upload()

In [ ]:
import json
from collections import Counter
from pathlib import Path

TRAIN_PATH = Path("train.jsonl")
EVAL_PATH = Path("eval.jsonl")

rows = [json.loads(line) for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
cases = evals.load_cases(EVAL_PATH)

sizes = Counter(len(r["messages"][-1]["content"].split(",")) for r in rows)
print(f"{len(rows)} training examples, {len(cases)} eval cases")
print("selection sizes:", dict(sorted(sizes.items())))

if len(rows) < 500:
    print(
        f"\nSTOP AND THINK: {len(rows)} examples is below MODEL.md §7's floor of 500.\n"
        "Few-shot prompting from the log (rung 2) will almost certainly beat a\n"
        "fine-tune on this much data, at zero cost. Keep using the daemon."
    )

# If one selection size dominates, the model learns the count instead of the
# criterion — it will look accurate on data drawn from the same distribution
# and fail the moment the right answer is 1 or 11.
top, n = sizes.most_common(1)[0]
if n / max(len(rows), 1) > 0.5:
    print(f"\nWARNING: {n/len(rows):.0%} of examples select exactly {top} items.")

---
## 3. Base model

`gemma-3-4b` is MODEL.md §4's starting recommendation for the curator tier — best-in-class memory efficiency at ~4.2 GB, permissive licence. **It is a candidate, not a commitment.** Swap it freely; the eval below is what decides.

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # candidate lists get long; lower this if VRAM is tight

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,  # the Q in QLoRA: 4-bit base, full-precision adapters
)
print(model.config.model_type)

---
## 4. Measure BEFORE training

This cell is the reason the notebook is worth running. Without a before number there is no after, and "it seems better" is not a result.

Two baselines are scored:

- **keep-all** — free, zero miss rate, 1× compression. This is what the curator must beat to justify existing at all.
- **base model, untuned** — what you already have without spending an afternoon.

In [ ]:
def make_runner(model, tokenizer, max_new_tokens=48):
    """Wrap a local HF model in the evals.Runner interface."""
    FastLanguageModel.for_inference(model)

    def _run(case):
        prompt = "\n".join(
            [f"TASK: {case.task}", "", "CANDIDATES:"]
            + [f"[{m.id}] {m.text}" for m in case.candidates]
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # selection is deterministic work
            pad_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

    return _run


before = [
    evals.run(cases, evals.keep_all_runner(), "baseline:keep-all"),
    evals.run(cases, make_runner(model, tokenizer), f"{BASE_MODEL} (untuned)"),
]
for report in before:
    print(report.render())
    print()

---
## 5. Attach the adapter

`r=16` per MODEL.md §7: rank 16 is for format and style adherence, which is exactly this task. `r=32` is general SFT; `r=64` is for complex multi-turn and coding work and is overkill for returning a list of integers.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # the main VRAM saving
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"training {trainable:,} of {total:,} params ({trainable/total:.2%})")

---
## 6. Train

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

dataset = Dataset.from_list(rows).map(
    lambda row: {
        "text": tokenizer.apply_chat_template(row["messages"], tokenize=False)
    }
)
print(dataset[0]["text"][:600])

In [ ]:
BATCH_SIZE = 2
GRAD_ACCUM = 4  # effective batch 8; raise this, not BATCH_SIZE, if VRAM is tight
EPOCHS = 2      # a narrow format task overfits fast — watch the loss, not the clock

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Loss on the assistant's answer only. Without this the model spends capacity
# learning to reproduce the candidate lists in the prompt — which is not the
# task, and on a selection job it is most of the tokens.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

stats = trainer.train()
print(f"\n{stats.metrics['train_runtime']/60:.1f} min")

---
## 7. Measure AFTER — and be willing to throw this away

The rule from MODEL.md §8: *any model change is a hypothesis, tested against this set before adoption. No exceptions, including for changes that are obviously better.*

In [ ]:
after = evals.run(cases, make_runner(model, tokenizer), f"{BASE_MODEL} + LoRA")
reports = before + [after]

for report in reports:
    print(report.render())
    print()
print(evals.compare(reports))

In [ ]:
untuned = next(r for r in before if "untuned" in r.label)
keep_all = next(r for r in before if "keep-all" in r.label)

print(f"penalty  untuned {untuned.penalty:.3f}  ->  tuned {after.penalty:.3f}")
print(f"miss     {untuned.miss_rate:.1%} -> {after.miss_rate:.1%}")
print(f"noise    {untuned.noise_rate:.1%} -> {after.noise_rate:.1%}")
print(f"format   {untuned.format_compliance:.1%} -> {after.format_compliance:.1%}")
print()

# 50 cases is enough to see a real difference and too few to trust a small one.
# A 2% move is noise; treat anything under ~5% as no result.
delta = untuned.penalty - after.penalty
if after.penalty >= keep_all.penalty:
    print("REJECT: does not beat keep-all. The curator is not earning its latency.")
elif delta < 0.05 * max(untuned.penalty, 1e-9):
    print("NO RESULT: within noise on 50 cases. Discard the adapter and go back")
    print("to rungs 1-3 (prompt, few-shot, better base). That is a successful run.")
else:
    print(f"IMPROVED by {delta:.3f}. Adopt it — and re-check on fresh cases later,")
    print("since this eval set is now something the adapter has been tuned against.")

---
## 8. Export — only if section 7 said IMPROVED

The adapter is tens of MB. GGUF export produces something Ollama serves directly, which is what the router's local backend points at.

In [ ]:
# The adapter alone — small, and composable with the base model at load time.
model.save_pretrained("curator-lora")
tokenizer.save_pretrained("curator-lora")
!zip -qr curator-lora.zip curator-lora
files.download("curator-lora.zip")

In [ ]:
# Merged GGUF for Ollama. Q4_K_M is MODEL.md §6's default — best quality per
# byte. Quantization damage shows up first as format drift, so re-run section 7
# against the quantized file before trusting it; the eval above scored fp16.
model.save_pretrained_gguf("curator-gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh curator-gguf/

### Serving it

Download the `.gguf`, then on the machine running Ollama:

```bash
printf 'FROM ./curator.gguf\nPARAMETER temperature 0\n' > Modelfile
ollama create one-for-all-curator -f Modelfile
```

Then add the name to the front of the curate role's `prefer` list in `~/.config/one-for-all/config.json` (`one-for-all-cli config --write` creates it). The router matches hints as substrings against the live model list, so it picks this up with no code change — and falls back gracefully to the generic size criteria if the model is ever absent.

Verify with:

```bash
one-for-all-cli status
one-for-all-cli eval eval.jsonl --compare-all
```

That second command is the one that matters — it scores the tuned model against every alternative through the real serving path, quantized, over HTTP, exactly as the daemon will call it.